# Step 2: Create Features (ATR)
This notebook calculates the Average True Range (ATR) for every stock in our tradable universe.

**Workflow:**
1. Load the raw OHLCV data (`top_5000_yf_data.pkl`)
2. Load the universe mask (`stores/universe_5m.parquet`) created by notebook 1
3. Calculate ATR using the vectorized function from `technical_indicators.py`
4. Mask the ATR so only in-universe stocks have values (out-of-universe → NaN)
5. Save to `stores/features.parquet` for downstream use

In [1]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))
from technical_indicators import calculate_atr_vectorized

import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "stores"
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
# Load raw OHLCV data
print('Loading top_5000_yf_data.pkl...')
pv = pd.read_pickle('top_5000_yf_data.pkl')
print(f'Data loaded. Shape: {pv.shape}')
print(f'Date range: {pv.index[0].date()} → {pv.index[-1].date()}')
print(f'Tickers: {len(pv["Close"].columns)}')

Loading top_5000_yf_data.pkl...
Data loaded. Shape: (4121, 30012)
Date range: 2010-01-04 → 2026-05-21
Tickers: 5002


In [3]:
# Load the tradable universe mask from notebook 1
universe_path = os.path.join(DATA_DIR, 'universe_5m.parquet')
df_universe = pd.read_parquet(universe_path)
print(f'Universe mask loaded. Shape: {df_universe.shape}')
print(f'Avg tradable stocks per day: {df_universe.sum(axis=1).mean():.0f}')

Universe mask loaded. Shape: (4117, 4999)
Avg tradable stocks per day: 1804


In [4]:
# Calculate ATR using vectorized function (runs in seconds)
print('Calculating ATR...')
indicators_dict = calculate_atr_vectorized(pv)
atr_all = indicators_dict['average_true_range']
print(f'Raw ATR shape: {atr_all.shape}')

Calculating ATR...
Raw ATR shape: (4121, 5002)


In [5]:
# Align ATR with the universe mask
# Only keep ATR values for stocks that are in the tradable universe on each day
# Out-of-universe stocks get NaN
common_dates = atr_all.index.intersection(df_universe.index)
common_tickers = atr_all.columns.intersection(df_universe.columns)

atr_aligned = atr_all.loc[common_dates, common_tickers]
universe_aligned = df_universe.loc[common_dates, common_tickers]

# Apply the mask: where universe == 0, set ATR to NaN
atr_masked = atr_aligned.where(universe_aligned == 1)

print(f'Masked ATR shape: {atr_masked.shape}')
print(f'Avg stocks with valid ATR per day: {atr_masked.notna().sum(axis=1).mean():.0f}')

Masked ATR shape: (4117, 5002)
Avg stocks with valid ATR per day: 1801


In [6]:
# Downcast to float32 to save memory
atr_masked = atr_masked.astype('float32')

# Preview
print('\nSample ATR values (last 5 days, first 10 in-universe stocks):')
last_day = atr_masked.iloc[-1]
valid_tickers = last_day.dropna().index[:10].tolist()
atr_masked[valid_tickers].tail()


Sample ATR values (last 5 days, first 10 in-universe stocks):


Ticker,A,AA,AAL,AAMI,AAOI,AAON,AAP,AAPL,AAT,AAUC
Date,,,,,,,,,,
2026-05-11,3.990715,2.238071,0.497857,2.736072,22.962711,9.445001,2.609285,6.637856,0.421429,0.991000
2026-05-12,3.968572,2.401858,0.497143,2.676071,23.960569,10.350715,2.627142,6.290715,0.445000,1.001715
2026-05-13,3.560001,2.340430,0.460000,2.646429,25.733427,10.810716,2.660714,6.526431,0.459286,1.035643
2026-05-14,3.607144,2.396645,0.453571,2.627142,25.510571,11.007144,2.604285,6.618574,0.468572,1.048143
2026-05-15,3.583572,2.574859,0.450000,2.846429,25.291285,11.007859,2.684285,6.667862,0.464286,0.910286


In [8]:
# Save as MultiIndex DataFrame under key 'average_true_range'
# This format is consistent with adding more features later
features_df = pd.concat({'average_true_range': atr_masked}, axis=1)
features_df = features_df.astype('float32')

# Drop any duplicate columns (like multiple 'NAN' tickers) before saving
features_df = features_df.loc[:, ~features_df.columns.duplicated()]

output_path = os.path.join(DATA_DIR, 'features.parquet')
features_df.to_parquet(output_path, compression='zstd', engine='pyarrow')
print(f'Saved to {output_path}')
print(f'Final features shape: {features_df.shape}')
print(f'Memory usage: {features_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

Saved to stores\features.parquet
Final features shape: (4117, 4999)
Memory usage: 82.4 MB
